In [1]:
import sys
sys.path.insert(0, '..')

import polars as pl
from diaphanous.show import show

pl.Config.set_thousands_separator(",")

%load_ext rpy2.ipython
import diaphanous.arr as arr
arr.install()
RLIB = arr.RLIB

import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
import rpy2.rinterface as ri

ro.globalenv['RLIB'] = RLIB
ro.r(".libPaths(RLIB)")

In [2]:
reports = pl.read_csv("../data/ocse-reports-per-year.csv")
population = pl.read_csv("../data/populations-simple.csv")
internet_users = pl.read_csv("../data/internet-users-un.csv")
social_media_accounts = pl.read_csv("../data/social-accounts.csv")

frame = reports.join(
    population, on="year", how="left"
).join(
    internet_users, on="year", how="left"
).join(
    social_media_accounts, on="year", how="left"
).select(
    pl.col("year", "reports", "population"),
    (pl.col("million_accounts") * 1_000_000).alias("accounts"),
    (pl.col("internet_users_pct") * pl.col("population") / 100).alias("internet_users"),
).filter(
    (pl.col("year") <= 2024) & (pl.col("year") >= 2014)
).drop_nans()

corrs = frame.corr().select(
    pl.col("internet_users", "population", "accounts")
).row(1)

data = frame.to_pandas()

show(data)

,year,reports,population,accounts,internet_users
0,"2,024","20,512,803","8,161,972,572.5","5,037,000,000.0","5,517,493,459.0"
1,"2,023","36,210,368","8,091,734,930.0","4,770,000,000.0","5,291,994,644.2"
2,"2,022","32,059,029","8,021,407,192.0","4,632,000,000.0","5,109,636,381.3"
3,"2,021","29,397,681","7,954,448,391.5","4,214,000,000.0","4,907,894,657.6"
4,"2,020","21,751,085","7,887,001,292.0","3,726,000,000.0","4,621,782,757.1"
5,"2,019","16,987,361","7,811,293,698.5","3,478,000,000.0","4,132,174,366.5"
6,"2,018","18,462,422","7,729,902,780.5","3,212,000,000.0","3,749,002,848.5"
7,"2,017","10,214,753","7,645,617,954.0","2,804,000,000.0","3,455,819,315.2"
8,"2,016","8,297,923","7,558,554,525.5","2,320,000,000.0","3,235,061,336.9"
9,"2,015","4,403,657","7,470,491,871.5","2,094,000,000.0","2,973,255,764.9"


In [3]:
stats = importr('stats')
base = importr('base')

with (ro.default_converter + pandas2ri.converter).context():
    rdata = ro.conversion.get_conversion().py2rpy(data)

ro.globalenv['data'] = rdata

ro.r("library(estimatr)")

r2_adj = {}
robust_r2_adj = {}

MODELS = [
    ('year', 'reports ~ year'),
    ('pop', 'reports ~ population'),
    ('inet', 'reports ~ internet_users'),
    ('social', 'reports ~ accounts'),
]

for name, relation in MODELS:
    ro.r(f'mod_{name} <- lm({relation}, data = data)')
    r2_adj[name] = ro.r(f'summary(mod_{name})$adj.r.squared')
    ro.r(f'robust_mod_{name} <- lm_robust({relation}, data = data)')
    robust_r2_adj[name] = ro.r(f'summary(robust_mod_{name})$adj.r.squared')


In [4]:
show("<h2>Normality</h2>")

for name, _ in MODELS:
    print(ro.r(f'shapiro.test(rstandard(mod_{name}))'))

show("<h2>Adjusted R²</h2>")
for key, value in sorted(r2_adj.items(), key=lambda i: i[1][0]):
    print(f"{key}: {value}")

show("<h3>Best Linear Model: Social Media Accounts</h3>")
print(ro.r('mod_social'))
print(ro.r('summary(mod_social)'))

show("<h2>Robust Adjusted R²</h2>")
for key, value in sorted(robust_r2_adj.items(), key=lambda i: i[1][0]):
    print(f"{key}: {value}")

show("<h3>Best Linear Model: Social Media Accounts</h3>")
print(ro.r('robust_mod_social'))
print(ro.r('summary(robust_mod_social)'))


	Shapiro-Wilk normality test

data:  rstandard(mod_year)
W = 0.82102, p-value = 0.01779



	Shapiro-Wilk normality test

data:  rstandard(mod_pop)
W = 0.84154, p-value = 0.03308



	Shapiro-Wilk normality test

data:  rstandard(mod_inet)
W = 0.85528, p-value = 0.05001



	Shapiro-Wilk normality test

data:  rstandard(mod_social)
W = 0.84612, p-value = 0.03797




year: [1] 0.7685491

pop: [1] 0.784028

inet: [1] 0.7976535

social: [1] 0.8019207




Call:
lm(formula = reports ~ accounts, data = data)

Coefficients:
(Intercept)     accounts  
 -1.413e+07    9.300e-03  



Call:
lm(formula = reports ~ accounts, data = data)

Residuals:
      Min        1Q    Median        3Q       Max 
-12200025  -1478569    853771   2917634   5980697 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept) -1.413e+07  5.238e+06  -2.698 0.024474 *  
accounts     9.300e-03  1.444e-03   6.441 0.000119 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 5085000 on 9 degrees of freedom
Multiple R-squared:  0.8217,	Adjusted R-squared:  0.8019 
F-statistic: 41.48 on 1 and 9 DF,  p-value: 0.0001194




year: [1] 0.7685491

pop: [1] 0.784028

inet: [1] 0.7976535

social: [1] 0.8019207



                 Estimate   Std. Error   t value    Pr(>|t|)      CI Lower
(Intercept) -1.413234e+07 5.703436e+06 -2.477864 0.035114292 -2.703441e+07
accounts     9.300212e-03 2.046176e-03  4.545167 0.001395555  4.671440e-03
                 CI Upper DF
(Intercept) -1.230270e+06  9
accounts     1.392898e-02  9


Call:
lm_robust(formula = reports ~ accounts, data = data)

Standard error type:  HC2 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)   CI Lower   CI Upper DF
(Intercept) -1.413e+07  5.703e+06  -2.478 0.035114 -2.703e+07 -1.230e+06  9
accounts     9.300e-03  2.046e-03   4.545 0.001396  4.671e-03  1.393e-02  9

Multiple R-squared:  0.8217 ,	Adjusted R-squared:  0.8019 
F-statistic: 20.66 on 1 and 9 DF,  p-value: 0.001396

